# Coffee Shop — Exploratory Data Analysis

Dataset: 149 116 transactions across 3 NYC locations (Jan–Jun 2023).  
Currency: Brazilian Real (R$), comma as decimal separator.

## 1. Setup & dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})
PALETTE = ['#6F4E37', '#C8A97E', '#3D2B1F', '#A0785A', '#E8D5B7']
sns.set_palette(PALETTE)

## 2. Load & clean

In [ ]:
CSV_PATH = 'assets/data/coffee-shop-dataset.csv'

def parse_brl(series: pd.Series) -> pd.Series:
    """Strip 'R$ ' prefix and convert BRL comma-decimal to float."""
    return (
        series
        .str.strip()
        .str.replace(r'^R\$\s*', '', regex=True)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )

raw = pd.read_csv(CSV_PATH)

df = raw.copy()
df['unit_price']  = parse_brl(df['unit_price'])
df['Total_Bill']  = parse_brl(df['Total_Bill'])
df['transaction_date'] = pd.to_datetime(df['transaction_date'], format='%d-%m-%Y')
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.time

print(f'Shape: {df.shape}')
print(f'Date range: {df.transaction_date.min().date()} → {df.transaction_date.max().date()}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

In [ ]:
# Sanity check: Total_Bill ≈ transaction_qty × unit_price
implied = (df.transaction_qty * df.unit_price).round(2)
mismatch = (implied != df.Total_Bill.round(2)).sum()
print(f'Bill vs qty×price mismatches: {mismatch} / {len(df)} rows')
df[['transaction_qty','unit_price','Total_Bill']].describe().round(2)

## 3. Revenue by hour — daily traffic pattern

In [ ]:
hourly = (
    df.groupby('Hour')
    .agg(transactions=('transaction_id','count'),
         revenue=('Total_Bill','sum'),
         avg_ticket=('Total_Bill','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(hourly.Hour, hourly.transactions / 1_000, color=PALETTE[0])
axes[0].set_title('Transactions by hour (thousands)')
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Transactions (k)')
axes[0].set_xticks(hourly.Hour)

axes[1].bar(hourly.Hour, hourly.revenue / 1_000, color=PALETTE[1])
axes[1].set_title('Revenue by hour (R$ thousands)')
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Revenue (R$ k)')
axes[1].set_xticks(hourly.Hour)

plt.tight_layout()
plt.savefig('assets/img/hourly_traffic.png', bbox_inches='tight')
plt.show()

print('\nPeak hour:', hourly.loc[hourly.transactions.idxmax(), 'Hour'], 'h')
print('Morning rush (6–10h):', f"{hourly[hourly.Hour <= 10].transactions.sum():,} txn")

## 4. Revenue by store

In [ ]:
store_summary = (
    df.groupby('store_location')
    .agg(transactions=('transaction_id','count'),
         revenue=('Total_Bill','sum'),
         avg_ticket=('Total_Bill','mean'))
    .sort_values('revenue', ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(store_summary.index, store_summary.revenue / 1_000, color=PALETTE[:3])
ax.bar_label(bars, fmt='R$ %.0fk', padding=4)
ax.set_title('Total revenue by store (Jan–Jun 2023)')
ax.set_xlabel('Revenue (R$ thousands)')
ax.set_xlim(0, store_summary.revenue.max() / 1_000 * 1.2)
plt.tight_layout()
plt.savefig('assets/img/store_revenue.png', bbox_inches='tight')
plt.show()

store_summary.assign(revenue_pct=lambda x: (x.revenue / x.revenue.sum() * 100).round(1))

## 5. Revenue by product category

In [ ]:
cat_summary = (
    df.groupby('product_category')
    .agg(revenue=('Total_Bill','sum'),
         transactions=('transaction_id','count'),
         avg_unit_price=('unit_price','mean'))
    .sort_values('revenue', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
top5 = cat_summary.head(5)
other_rev = cat_summary.revenue.iloc[5:].sum()
pie_data = pd.concat([top5.revenue, pd.Series({'Other': other_rev})])
axes[0].pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
            colors=PALETTE + ['#d3d3d3'], startangle=140)
axes[0].set_title('Revenue share by category')

# Bar — avg unit price
axes[1].barh(cat_summary.index[::-1], cat_summary.avg_unit_price[::-1], color=PALETTE[2])
axes[1].set_title('Avg unit price by category (R$)')
axes[1].set_xlabel('Avg unit price (R$)')

plt.tight_layout()
plt.savefig('assets/img/category_breakdown.png', bbox_inches='tight')
plt.show()

cat_summary.round(2)

## 6. Monthly growth trend

In [ ]:
monthly = (
    df.groupby(['Month', 'Month Name'])
    .agg(revenue=('Total_Bill','sum'),
         transactions=('transaction_id','count'))
    .reset_index()
    .sort_values('Month')
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(monthly['Month Name'], monthly.revenue / 1_000, marker='o',
        linewidth=2.5, color=PALETTE[0], markersize=7)
ax.fill_between(range(len(monthly)), monthly.revenue / 1_000,
                alpha=0.12, color=PALETTE[0])
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Month Name'])
ax.set_title('Monthly revenue trend (R$ thousands)')
ax.set_ylabel('Revenue (R$ k)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:.0f}k'))
plt.tight_layout()
plt.savefig('assets/img/monthly_trend.png', bbox_inches='tight')
plt.show()

r_month, p_month = stats.pearsonr(monthly.Month, monthly.revenue)
print(f'Pearson r(month, revenue) = {r_month:.4f}  |  p = {p_month:.4e}')
print('Interpretation: strong monotonic growth — revenue nearly doubles from Jan to Jun.')

## 7. Correlation: syrup/flavour ratio × coffee avg ticket by hour  (r ≈ 0.74)

**Hypothesis from the executive summary:** hours with a higher share of syrup/flavour orders
also show a higher average coffee ticket — customers who customise tend to spend more.

- **X:** proportion of Flavours orders among (Flavours + Coffee) transactions in that hour
- **Y:** average `Total_Bill` for Coffee transactions in that hour

Computed at the hourly level (15 points, hours 6–20).

In [ ]:
# Hourly aggregation split by category
hourly_cat = (
    df[df.product_category.isin(['Flavours', 'Coffee'])]
    .groupby(['Hour', 'product_category'])
    .agg(txn_count=('transaction_id', 'count'),
         avg_ticket=('Total_Bill', 'mean'))
    .unstack('product_category')
)
hourly_cat.columns = ['_'.join(c) for c in hourly_cat.columns]
hourly_cat = hourly_cat.reset_index().dropna()

# Syrup ratio = Flavours txn / (Flavours + Coffee txn) per hour
hourly_cat['syrup_ratio'] = (
    hourly_cat['txn_count_Flavours']
    / (hourly_cat['txn_count_Flavours'] + hourly_cat['txn_count_Coffee'])
)

r_syrup, p_syrup = stats.pearsonr(hourly_cat.syrup_ratio, hourly_cat.avg_ticket_Coffee)
print(f'r(syrup_ratio_by_hour, coffee_avg_ticket_by_hour) = {r_syrup:.4f}  |  p = {p_syrup:.4f}')
hourly_cat[['Hour', 'syrup_ratio', 'avg_ticket_Coffee']].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(
    hourly_cat.syrup_ratio * 100,
    hourly_cat.avg_ticket_Coffee,
    c=hourly_cat.Hour, cmap='YlOrBr', s=90, zorder=3
)
plt.colorbar(sc, ax=ax, label='Hour of day')

m, b = np.polyfit(hourly_cat.syrup_ratio, hourly_cat.avg_ticket_Coffee, 1)
x_range = np.linspace(hourly_cat.syrup_ratio.min(), hourly_cat.syrup_ratio.max(), 100)
ax.plot(x_range * 100, m * x_range + b, color='#6F4E37', linewidth=2,
        label=f'r = {r_syrup:.2f}')

for _, row in hourly_cat.iterrows():
    ax.annotate(f"{int(row.Hour)}h",
                (row.syrup_ratio * 100, row.avg_ticket_Coffee),
                textcoords='offset points', xytext=(5, 3), fontsize=8, color='#3D2B1F')

ax.set_xlabel('Syrup/Flavour orders as % of (Flavours + Coffee)')
ax.set_ylabel('Avg Coffee ticket (R$)')
ax.set_title('Syrup ratio vs Coffee avg ticket — by hour of day')
ax.legend()
plt.tight_layout()
plt.savefig('assets/img/correlation_syrup_ticket.png', bbox_inches='tight')
plt.show()

print('Off-peak hours (6h, 20h): highest syrup ratio AND highest avg ticket.')
print('Rush hours (10–14h): plain coffee dominates → ratio and ticket both drop.')
print('Implication: upselling syrups during peak hours is the highest-ROI lever.')

## 8. Store × Hour heatmap

In [ ]:
pivot = (
    df.groupby(['store_location', 'Hour'])['Total_Bill']
    .sum()
    .unstack('Hour')
    / 1_000
)

fig, ax = plt.subplots(figsize=(13, 3))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrBr',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Revenue (R$ k)'})
ax.set_title('Revenue heatmap: store × hour (R$ thousands)')
ax.set_xlabel('Hour of day')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('assets/img/heatmap_store_hour.png', bbox_inches='tight')
plt.show()

In [ ]:
total_revenue = df.Total_Bill.sum()
avg_daily = df.groupby('transaction_date').Total_Bill.sum().mean()
peak_hour = hourly.loc[hourly.transactions.idxmax(), 'Hour']
top_category = cat_summary.index[0]
top_category_share = cat_summary.revenue.iloc[0] / total_revenue

print('=== Key Metrics ===')
print(f'Total revenue        : R$ {total_revenue:,.2f}')
print(f'Total transactions   : {len(df):,}')
print(f'Avg daily revenue    : R$ {avg_daily:,.2f}')
print(f'Avg ticket size      : R$ {df.Total_Bill.mean():.2f}')
print(f'Peak hour            : {peak_hour}h')
print(f'Top category         : {top_category} ({top_category_share:.1%} of revenue)')
print(f'Monthly growth (r)   : {r_month:.4f}  (near-linear ramp Jan→Jun)')
print(f'Syrup→ticket corr (r): {r_syrup:.4f}  (hours with more syrup orders = higher coffee ticket)')

In [ ]:
total_revenue = df.Total_Bill.sum()
avg_daily = df.groupby('transaction_date').Total_Bill.sum().mean()
peak_hour = hourly.loc[hourly.transactions.idxmax(), 'Hour']
top_category = cat_summary.index[0]
top_category_share = cat_summary.revenue.iloc[0] / total_revenue

print('=== Key Metrics ===')
print(f'Total revenue        : R$ {total_revenue:,.2f}')
print(f'Total transactions   : {len(df):,}')
print(f'Avg daily revenue    : R$ {avg_daily:,.2f}')
print(f'Avg ticket size      : R$ {df.Total_Bill.mean():.2f}')
print(f'Peak hour            : {peak_hour}h')
print(f'Top category         : {top_category} ({top_category_share:.1%} of revenue)')
print(f'Monthly growth (r)   : {r_month:.4f}  (near-linear ramp Jan→Jun)')
print(f'Price→Bill corr (r)  : {r_bill_price:.4f}  (unit price explains ~48% of bill variance)')

## 9. Simulated cohort — category revenue trajectory (Jan = 100)

Without a customer ID, a true retention cohort is impossible. The workaround: treat each
**product category** as a cohort and index its monthly revenue to January = 100.
This reveals which categories are outpacing the market and which are decoupling.

- **Index > 100** → category is above its own January baseline
- **Index < 100** → category revenue fell below where it started

In [ ]:
# Monthly revenue by category, indexed to January = 100
monthly_cat = (
    df.groupby(['Month', 'product_category'])['Total_Bill']
    .sum()
    .unstack('product_category')
    .sort_index()
)

cohort = (monthly_cat / monthly_cat.iloc[0] * 100).round(1)
cohort.index = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

print("Category revenue indexed to Jan = 100\n")
print(cohort.to_string())
print(f"\nSemester range: min={cohort.min().min():.0f}  max={cohort.max().max():.0f}")
print(f"Branded Jun index : {cohort['Branded'].iloc[-1]:.0f}  (market avg: {cohort.drop(columns='Branded').iloc[-1].mean():.0f})")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

im = ax.imshow(cohort.T.values, cmap='RdYlGn', vmin=60, vmax=220, aspect='auto')
plt.colorbar(im, ax=ax, label='Index (Jan = 100)', shrink=0.8)

month_labels = cohort.index.tolist()
cat_labels = cohort.columns.tolist()

ax.set_xticks(range(len(month_labels)))
ax.set_xticklabels(month_labels)
ax.set_yticks(range(len(cat_labels)))
ax.set_yticklabels(cat_labels)

for i, cat in enumerate(cat_labels):
    for j, month in enumerate(month_labels):
        val = cohort.loc[month, cat]
        text_color = 'black' if 80 < val < 180 else 'white'
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                fontsize=8.5, fontweight='bold', color=text_color)

ax.set_title('Category revenue cohort — indexed to Jan = 100', pad=12)
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig('assets/img/cohort_categories.png', bbox_inches='tight')
plt.show()

print("Red cells (< 100): revenue below January baseline")
print("Green cells (> 100): revenue above January baseline")
print("Branded Feb = 65 is the only cell below 100 in the entire heatmap.")

In [ ]:
# Incremental revenue decomposition: how much of June is above the Jan baseline?
jan_daily = df[df['Month'] == 1].groupby('transaction_date')['Total_Bill'].sum().mean()
jun_daily = df[df['Month'] == 6].groupby('transaction_date')['Total_Bill'].sum().mean()
incremental_pct = (jun_daily - jan_daily) / jun_daily * 100

print("=== Incremental Revenue Decomposition ===")
print(f"Jan avg daily revenue  : R$ {jan_daily:,.2f}  (baseline)")
print(f"Jun avg daily revenue  : R$ {jun_daily:,.2f}")
print(f"Incremental above base : {incremental_pct:.1f}% of June revenue\n")

# Branded vs market comparison across the semester
market_avg = cohort.drop(columns='Branded').mean(axis=1).round(1)
gap = (cohort['Branded'] - market_avg).round(1)

comparison = pd.DataFrame({
    'Branded index': cohort['Branded'],
    'Market avg (excl. Branded)': market_avg,
    'Gap (pp)': gap
})
print("=== Branded vs Market Average ===")
print(comparison.to_string())
print(f"\nBranded underperformed the market by {abs(gap.iloc[-1]):.0f} pp in June.")
print("Loose Tea fastest grower:", f"index {cohort['Loose Tea'].iloc[-1]:.0f} in June.")